### DNS Log Event GeoMap
Part of the DUNE project for ML-enhanced threat hunting [https://github.com/opendr-io/dune]

Ingests DNS events from an Elasticsearch export and plots them on a map. The last cell can be used to view raw events.

In [1]:
# 1. Settings
LOG_DIR = "logs"      # folder with the DNS log CSVs to analyze (Kibana exports with ECS field names); every *.csv in it is loaded
DEDUPLICATE = True    # True = drop rows whose fields all match an earlier row, within a file or across overlapping exports

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
from babel import Locale   # installed with Jupyter; supplies English country names
from babel.core import get_global

In [3]:
# 2. Ingest
# Columns the later cells use
REQUIRED = ['@timestamp', 'source.ip', 'destination.ip', 'dns.question.name',
            'dns.answers.name', 'dns.type', 'dns.question.type']

frames = []
for path in sorted(Path(LOG_DIR).glob("*.csv")):
    log = pd.read_csv(path)
    missing = [col for col in REQUIRED if col not in log.columns]
    if missing:
        print(f"Skipping {path.name}: missing columns {missing}")
        continue
    frames.append(log.assign(log_file=path.name))
if not frames:
    raise FileNotFoundError(f"No usable DNS log CSVs in {Path(LOG_DIR).resolve()}")

df = pd.concat(frames, ignore_index=True)
# Kibana writes '-' for empty fields; treat blank cells the same way
df[REQUIRED] = df[REQUIRED].fillna('-')

# Drop rows whose fields all match an earlier row (log_file isn't compared, so the first file's copy is kept)
if DEDUPLICATE:
    before = len(df)
    df = df.drop_duplicates(subset=df.columns.drop('log_file'), ignore_index=True)
    print(f"Dropped {before - len(df)} duplicate rows")
print(f"Loaded {len(df)} events from {len(frames)} file(s)")

Dropped 2034 duplicate rows
Loaded 18433 events from 2 file(s)


In [4]:
# 3. Suffix -> country mapping
# Two-letter suffixes are country codes; Babel has the English name for each (RU -> Russia, CN -> China)
TERRITORIES = Locale('en').territories
# Babel region codes that aren't country-code suffixes (groupings, test codes, regions without their own suffix)
NOT_CCTLDS = {'EZ', 'UN', 'QO', 'XA', 'XB', 'XK', 'ZZ', 'EA', 'IC', 'DG', 'CP', 'TA', 'CQ', 'BL', 'BQ', 'MF', 'EH'}
# Country-code suffixes that aren't ISO country codes
EXTRA_CCTLDS = {'uk': 'United Kingdom', 'su': 'Soviet Union (legacy)'}
# Country codes widely sold as generic names (.io, .co, .me ...), so the country says little about who runs the domain
VANITY_CCTLDS = {'ai', 'cc', 'co', 'fm', 'gg', 'io', 'ly', 'me', 'sh', 'tk', 'to', 'tv', 'ws'}
# Common generic suffixes, not tied to a country
GENERIC_TLDS = {'com', 'net', 'org', 'edu', 'gov', 'mil', 'int', 'info', 'biz', 'name', 'pro', 'mobi', 'aero',
                'asia', 'coop', 'jobs', 'museum', 'tel', 'travel', 'app', 'dev', 'top', 'xyz', 'site', 'online',
                'store', 'shop', 'club', 'live', 'tech', 'cloud', 'goog', 'google', 'amazon', 'microsoft', 'apple'}
# Names that never reach the public DNS: reserved, internal or server-information names (e.g. version.bind)
SPECIAL_USE = {'arpa', 'local', 'localhost', 'localdomain', 'internal', 'invalid', 'test', 'example', 'onion',
               'home', 'lan', 'corp', 'intranet', 'private', 'workgroup', 'bind'}

def suffix_info(name):
    # (suffix, category, country) for one query name; country is None unless the suffix is a country code
    name = str(name).strip().rstrip('.').lower()
    if name in ('', '-'):
        return '-', 'missing', None
    labels = name.split('.')
    suffix = labels[-1]
    if suffix.isdigit():
        return suffix, 'IP address / numeric', None   # e.g. a query for 54.91.63.29
    if suffix in EXTRA_CCTLDS:
        country = EXTRA_CCTLDS[suffix]
    elif len(suffix) == 2 and suffix.isalpha() and suffix.upper() not in NOT_CCTLDS:
        country = TERRITORIES.get(suffix.upper())
    else:
        country = None
    if country:
        return suffix, 'country code (often used generically)' if suffix in VANITY_CCTLDS else 'country code', country
    if suffix in SPECIAL_USE:
        return suffix, 'special-use / internal', None
    if suffix in GENERIC_TLDS:
        return suffix, 'generic', None
    if any(ch.isdigit() for ch in suffix) or ('-' in suffix and not suffix.startswith('xn--')):
        return suffix, 'internal host name', None   # public suffixes have no digits or hyphens (except xn-- names)
    if len(labels) == 1:
        return suffix, 'single-label host name', None   # e.g. 'server' or 'ip-172-16-200-110'
    return suffix, 'other (probably a newer generic suffix)', None

df[['suffix', 'category', 'country']] = [suffix_info(name) for name in df['dns.question.name']]
print(df['category'].value_counts().to_string())

category
generic                                    16867
special-use / internal                      1077
country code                                 241
internal host name                           176
other (probably a newer generic suffix)       38
country code (often used generically)         29
missing                                        5


In [5]:
# 4. Count domain suffixes: events, distinct query names and distinct callers per suffix
suffixes = df.groupby('suffix').agg(
    category=('category', 'first'),
    country=('country', 'first'),
    events=('suffix', 'size'),
    query_names=('dns.question.name', 'nunique'),
    source_ips=('source.ip', 'nunique'),
).sort_values('events', ascending=False)


In [6]:
# 5. Countries: the country-code suffixes seen, with events, distinct query names and distinct callers
countries = df[df['country'].notna()].groupby('country').agg(
    suffixes=('suffix', lambda s: ', '.join(sorted(set(s)))),
    events=('suffix', 'size'),
    query_names=('dns.question.name', 'nunique'),
    source_ips=('source.ip', 'nunique'),
).sort_values('events', ascending=False)

print(f"{len(countries)} countries, {countries['events'].sum()} events with a country-code suffix")
with pd.option_context('display.max_rows', None):
    display(countries)

12 countries, 270 events with a country-code suffix


,suffixes,events,query_names,source_ips
country,,,,
Sierra Leone,sl,217,1,60
British Indian Ocean Territory,io,14,3,7
Colombia,co,13,2,2
Russia,ru,6,1,1
Serbia,rs,5,5,1
France,fr,4,1,1
European Union,eu,3,1,2
Luxembourg,lu,3,1,3
Argentina,ar,2,1,1


In [7]:
# 6. Map: events by country, shaded on a world map (hover over a country for its counts)
# Three-letter codes for the map (RU -> RUS) from Babel's data; skip retired codes that share a
# two-letter code (e.g. FXX for FR) and Q../X.. codes, which aren't countries (e.g. QUU for the EU)
RETIRED = {'BUR', 'ZAR', 'DDR', 'FXX', 'TMP', 'YMD'}
ALPHA3 = {two[0]: three for three, two in get_global('territory_aliases').items()
          if len(three) == 3 and three.isalpha() and len(two) == 1
          and three not in RETIRED and three[0] not in 'QX'}

# Each country's code comes from its suffix; .uk is the exception, its ISO code is GB
iso3 = (df[df['country'].notna()]
        .assign(iso3=lambda d: d['suffix'].map(lambda s: ALPHA3.get('GB' if s == 'uk' else s.upper())))
        .groupby('country')['iso3'].first())
map_data = countries.join(iso3).reset_index()
on_map = map_data.dropna(subset=['iso3']).copy()

# Colour by log10(events) so one very busy country doesn't wash out the rest; the colour bar shows real counts
on_map['log_events'] = np.log10(on_map['events'])
ticks = [10 ** k for k in range(int(np.log10(on_map['events'].max())) + 1)]

fig = px.choropleth(on_map, locations='iso3', color='log_events', hover_name='country',
                    hover_data={'suffixes': True, 'events': True, 'query_names': True, 'source_ips': True,
                                'iso3': False, 'log_events': False},
                    color_continuous_scale='Reds', title='DNS events by country-code suffix')
fig.update_coloraxes(colorbar=dict(title='Events', tickvals=np.log10(ticks), ticktext=[f'{t:,}' for t in ticks]))
fig.update_geos(resolution=50, showcountries=True, countrycolor='lightgray', projection_type='natural earth')
fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

# Suffixes that aren't a single country (e.g. .eu, .su) can't be drawn
not_drawn = map_data[map_data['iso3'].isna()]
if len(not_drawn):
    print("Not on the map: " + ", ".join(f"{row.country} ({row.events} events)" for row in not_drawn.itertuples()))

Not on the map: European Union (3 events)


In [8]:
# 7. Look up the DNS events for one country: pick it from the list (busiest first, events in brackets)
country_box = widgets.Dropdown(options=[(f"{country} ({row.events})", country) for country, row in countries.iterrows()],
                               value=None, description='Country:', layout=widgets.Layout(width='400px'))
country_output = widgets.Output()

def show_country(_=None):
    with country_output:
        country_output.clear_output()
        if country_box.value is None:
            print("Pick a country to see its events.")
            return
        rows = df[df['country'] == country_box.value]
        print(f"{len(rows)} events, {rows['dns.question.name'].nunique()} distinct query names, "
              f"{rows['source.ip'].nunique()} source IPs")
        # Every row and every column, instead of pandas' defaults that hide some behind '...'
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(rows)

country_box.observe(show_country, names='value')
show_country()
display(widgets.VBox([country_box, country_output]))